# IndoNewsSum — Kaggle Training Run

## Setup

1. **Settings → Accelerator** → `GPU T4 x2`
2. **Settings → Internet** → **ON**
3. **Add-ons → Secrets** → add a secret with key `HF_TOKEN` and value = your Hugging Face access token

## Run cells in order:

| Cell | Action |
|------|--------|
| 1    | Clone the repository |
| 2    | Install Kaggle-safe Unsloth stack |
| 3    | Inject HF_TOKEN from Kaggle Secrets |
| 4    | Train (≈ 6 h on 2× T4) |
| 5    | Evaluate ROUGE on 50 val samples |
| 6    | Export merged model (+ optional GGUF) |

Once done, deploy the Gradio Space from `space/app.py` pointing at the HF repo.

In [ ]:
# Clone repository
!git clone https://github.com/icaluwu/Indonesian-News-Summarizer.git /kaggle/working/indo-news-summarizer
%cd /kaggle/working/indo-news-summarizer

print("Cloned to", os.getcwd())


In [ ]:
# Kaggle-safe Unsloth install (pre-installed dependencies, no re-download)
%pip install --no-deps bitsandbytes accelerate xformers==0.0.30 peft trl triton unsloth_zoo
%pip install --no-deps unsloth


In [ ]:
# Load HF_TOKEN from Kaggle Secrets
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")
print("HF_TOKEN loaded from Kaggle Secrets:", "***")


In [ ]:
# Edit configs/train.yaml first if hf_repo_id is still null
!python src/train.py --config configs/train.yaml


In [ ]:
!python src/evaluate.py --config configs/train.yaml --adapter outputs/adapter --n 50


In [ ]:
!python src/export.py --config configs/train.yaml --adapter outputs/adapter --out outputs/merged --gguf


## Deploy Gradio Space

Push the `space/` directory to a Hugging Face Space:

1. Go to https://huggingface.co/spaces
2. **Create new Space** → name: `Indonesia-News-Summarizer-Qwen3.8-27b`
3. Hardware: **Custom** → `A10G` (or any GPU with ≥ 16 GB VRAM)
4. Upload `space/app.py` and `space/requirements.txt`
5. In Space **Settings → Secrets**, add `HF_TOKEN`

The Space loads the LoRA adapter from `icaluwu/Indonesia-News-Summarizer-Qwen3.8-27b` on startup.